In [1]:
!pip install bertopic sentence-transformers umap-learn hdbscan

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 1.2 MB/s  0:00:05 eta 0:00:010m
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 896.2 kB/s  0:00:12m0:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 625.2/625.2 kB 1.2 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 1.6 MB/s  0:00:02 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 1.6 MB/s  0:00:02 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 30.0 MB/s  0:00:01 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 91.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 92.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 801.6/801.6 kB 63.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 802.1/802.1 kB 40.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
# Si hdbscan marche pas on peut utiliser ça

from sklearn.cluster import HDBSCAN
from bertopic.cluster import BaseCluster
"""
class SklearnHDBSCAN(BaseCluster):
    def __init__(self, *args, **kwargs):
        self.model = HDBSCAN(*args, **kwargs)
    
    def fit(self, X):
        self.model.fit(X)
        self.labels_ = self.model.labels_
        return self
    
    def fit_predict(self, X):
        self.labels_ = self.model.fit_predict(X)
        return self.labels_
hdbscan_model = SklearnHDBSCAN(
    min_cluster_size=50,      # bigger = fewer, larger topics
    min_samples=10,            # higher = more conservative clustering
    cluster_selection_epsilon=0.1,  # merge nearby clusters
    cluster_selection_method='leaf' # fewer, more specific clusters
)
"""

Récupération des fichiers CSV

In [ ]:
import pandas as pd
import os
import s3fs



fs = s3fs.S3FileSystem(
    client_kwargs={'endpoint_url': 'https://'+'minio.lab.sspcloud.fr'},
    key = os.environ["AWS_ACCESS_KEY_ID"], 
    secret = os.environ["AWS_SECRET_ACCESS_KEY"], 
    token = os.environ["AWS_SESSION_TOKEN"])

s3_folder = "biath"

files = fs.glob(f"s3://lab/PESSD/csv/{s3_folder}/*.csv")

for file in files:
    fs.get(file, "data/"+(file.removeprefix(f"lab/PESSD/csv/{s3_folder}")))

In [31]:
df = pd.DataFrame()
for file in os.listdir("data") : 
    temp = pd.read_csv("data/"+file)
    df = pd.concat([df,temp])


Nettoyage de la data

In [ ]:
import re
import string

# Removing music and noise (only technical commentary)
df = df[(df["main_g"]=="female")|(df["main_g"]=="male")]

# text to sentences
df["text"] = df["text"].apply(lambda x : x.split(". "))
df = df.explode("text")

# Removes all words with capitals
df["text"] = df["text"].apply(lambda x : re.sub(r"\s*[A-Z]\w*\s*", " ", x).strip())

# Remove punctuation
df ["text"] = df["text"].apply(lambda x : x.translate(str.maketrans('', '', string.punctuation.replace("'", ""))))

On ajoute les métadonnées

In [33]:
df["ID"] = df["audio_file"].str.replace("_wav.wav","")
df = df.merge(pd.read_csv("metadata.csv"), on="ID")

In [18]:
df

,start,stop,text,main_speaker,main_g,audio_file,ID,g_ath
0,0.00,35.64,'est la question,SPEAKER_03,male,BIF_wav.wav,BIF,F
1,0.00,35.64,a des favorites,SPEAKER_03,male,BIF_wav.wav,BIF,F
2,0.00,35.64,problème c'est de pouvoir confirmer le jour,SPEAKER_03,male,BIF_wav.wav,BIF,F
3,0.00,35.64,aujourd'hui est un jour important pour les ath...,SPEAKER_03,male,BIF_wav.wav,BIF,F
4,0.00,35.64,qui a fait notamment un podium en ouverture de...,SPEAKER_03,male,BIF_wav.wav,BIF,F
...,...,...,...,...,...,...,...,...
8839,2893.44,2919.36,il y a - qui est là,SPEAKER_02,male,BMH_wav.wav,BMH,H
8840,2893.44,2919.36,- on va garder ce chiffre en tête en tête à r...,SPEAKER_02,male,BMH_wav.wav,BMH,H
8841,2893.44,2919.36,devient de toute l'histoire des le français l...,SPEAKER_02,male,BMH_wav.wav,BMH,H
8842,2893.44,2919.36,sacré cours et sacré jeu pour,SPEAKER_02,male,BMH_wav.wav,BMH,H


In [37]:
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from umap import UMAP
import hdbscan

# Stopwords list
STOPWORDS = [x.strip() for x in open('stopwords.txt').readlines()]

# Docs 
docs = df["text"]

# Vectorizer for topic representation
vectorizer_model = CountVectorizer(
    stop_words=STOPWORDS,
    ngram_range=(1,2),
    min_df=2
    #token_pattern=r"\b(?!\w*[A-Z])\w+\b" # theoreticaly removes words with capital letters once embeddings are done
)

# To control for number and size of clusters
hdbscan_model = hdbscan.HDBSCAN(
    min_cluster_size=50,      # bigger = fewer, larger topics
    min_samples=10,            # higher = more conservative clustering
    cluster_selection_epsilon=0.1,  # merge nearby clusters
    cluster_selection_method='leaf', # fewer, more specific clusters
    prediction_data=True # important !!
)

# Load CamemBERT embedding model
embedding_model = SentenceTransformer("dangvantuan/sentence-camembert-base", device="cuda") # remove device if CPU

# Influence choice of themes
seed =  [
["émotion","larme","ému","joie","tristesse"],
["famille","parents","ami", "personnel", "conjoint"]]

# Build BERTopic model
topic_model = BERTopic(
    embedding_model=embedding_model,
    vectorizer_model=vectorizer_model,
    hdbscan_model=hdbscan_model,
    language="french",
    calculate_probabilities=True,
    verbose=True,
    seed_topic_list=None    
)

# Fit the model
topics, probs = topic_model.fit_transform(docs)

# Show discovered topics
print(topic_model.get_topic_info())

# Visualize topics
#topic_model.visualize_topics()

# Visualize topic hierarchy
topic_model.visualize_hierarchy()

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3134.20it/s]
CamembertModel LOAD REPORT from: dangvantuan/sentence-camembert-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
2026-03-27 19:14:47,831 - BERTopic - Embedding - Transforming documents to embeddings.
Batches: 100%|██████████| 269/269 [00:09<00:00, 28.38it/s]
2026-03-27 19:14:57,403 - BERTopic - Embedding - Completed ✓
2026-03-27 19:14:57,404 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-03-27 19:15:54,025 - BERTopic - Dimensionality - Completed ✓
2026-03-27 19:15:54,030 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-03-27 19:15:55,643 - BERTopic - Cluster - Completed ✓
2026-03-27 19:15:55,656 - BERTopic - Representation - Fine-tuning topics using representatio

    Topic  Count                                               Name  \
0      -1   2979                -1_secondes_monde_podium_individuel   
1       0    578                       0_tir_cible_balle_sortie tir   
2       1    274                           1_position_10_tête_10 10   
3       2    242                 2_attention_aïe_présents_également   
4       3    235                     3_médaille_médailles_or_argent   
5       4    232                        4_skis_ski_temps ski_rapide   
6       5    177                    5_équipe_filles_athlètes_groupe   
7       6    167                       6_secondes_10 secondes_50_10   
8       7    165                     7_montée_bosse_altitude_glisse   
9       8    155             8_relance_recoller_repris_repris temps   
10      9    143                                9_20_20 20_19_19 20   
11     10    136        10_coucher_réussite_couché_réussite coucher   
12     11    121                     11_faut_relais_vraiment_course   
13    

On regroupe les phrases par topic

In [ ]:
"""tp = pd.DataFrame({
    "text": docs,
    "topic": topics,
    "topic_prob": probs.max(axis=1)  # max probability per text
})"""

In [47]:
#df = df.merge(tp, on="text")
df["topic"] = topics
df["topic_prob"] = probs.max(axis=1)
df

,start,stop,text,main_speaker,main_g,audio_file,ID,g_ath,topic,topic_prob
0,0.00,35.64,'est la question,SPEAKER_03,male,BIF_wav.wav,BIF,F,31,0.296411
1,0.00,35.64,a des favorites,SPEAKER_03,male,BIF_wav.wav,BIF,F,-1,0.014570
2,0.00,35.64,problème c'est de pouvoir confirmer le jour,SPEAKER_03,male,BIF_wav.wav,BIF,F,22,1.000000
3,0.00,35.64,aujourd'hui est un jour important pour les ath...,SPEAKER_03,male,BIF_wav.wav,BIF,F,-1,0.043955
4,0.00,35.64,qui a fait notamment un podium en ouverture de...,SPEAKER_03,male,BIF_wav.wav,BIF,F,-1,0.020902
...,...,...,...,...,...,...,...,...,...,...
8839,2893.44,2919.36,il y a - qui est là,SPEAKER_02,male,BMH_wav.wav,BMH,H,31,1.000000
8840,2893.44,2919.36,- on va garder ce chiffre en tête en tête à r...,SPEAKER_02,male,BMH_wav.wav,BMH,H,26,0.343090
8841,2893.44,2919.36,devient de toute l'histoire des le français l...,SPEAKER_02,male,BMH_wav.wav,BMH,H,-1,0.045265
8842,2893.44,2919.36,sacré cours et sacré jeu pour,SPEAKER_02,male,BMH_wav.wav,BMH,H,39,0.116769


In [60]:
# Sentences of topic i

for k in df[df["topic"]==2]["text"] : 
    print(k)

'un des plus beaux avec   avec ce qui est en train de devenir le   avec aussi  qui est un endroit particulier
 salut  salut
même  -  dont on parle parfois peu
regardez on en parlait
et puis 
norvégienne
est installée du côté de l'
tout petit gabarit  
alors   et  -
sont présents
pour  - 
qui continue
elle déclenche
avec des absences sur la du
rivalise avec
au profit de
a l'instar de
 punaise
dernier passage
est condamnée
craque complètement  
là là c'est mon rage pour  comme sur l'oreillet mixte
 le titre par  la voilà
course de
course de ici en -
surtout dans cette zone oui
est sur la
derrière  -
et puis
 suit le map
incroyable le cours de
derrière déjà 
pas de réaction du côté d'
adore le classique
peut l'être
 allez les 
sont bonnes salut  salut à tous
avis aussi sur
y a eu beaucoup de
là l'  la  les États-
 il y a les avec et
a déjà eu des configurations
 il y a
a été un prévue italien staneur sur ces
 allez-y
on a qui  a  
la bosse
a complé lui aussi
et dès il y a
'américain
relai

In [59]:
# Sentences of topic i and by sport

for _, j in df[df["topic"]==34][["text","ID"]].iterrows():
    print(j["ID"],"_", j["text"])

BIF _  c'est un peu disparu des radars
BIF _ le soleil c'est un petit peu caché derrière les nuages
BIF _ sera même derrière 
BIF _ est bien derrière même
BIF _ s'est reprise derrière
BIF _ est passée derrière
BIF _ Ça sera derrière 
BIF _ mais c'est derrière    c'est derrière
BIF _ Ça sera derrière comme  également
BIF _ Ça sera derrière 
BIF _ Ça sera derrière pour l'
BAH _ intelligent derrière
BAH _ puis derrière c'est et  les 
BAH _ serait trop cruel là qu'il repasse de nouveau derrière
BAH _ est à 5 mètres derrière
BAH _ prend 5 mètres comme ça ça veut dire que quand même c'est dans le dur
BAH _ a derrière
BAH _ il reste il reste encore du monde derrière rien n'est fait pour l'instant ouais on a vu sur le close-off
BAH _ 'est un truc de l'ombre
BAH _  il met derrière
BAH _ là la réalité c'est quoi 'est que par contre derrière c'est moi
BAH _ n'y a plus personne derrière
BSF _ a vu -  derrière la jumelle
BSF _ Ça sera trop pour arriver dans les temps et elle sera derrière malheureu

In [58]:
topic_model.get_topic(34)

[('abri', np.float64(0.09498799679276133)),
 ('mètres', np.float64(0.07693232140076829)),
 ('met', np.float64(0.07411948227123588)),
 ('instant', np.float64(0.07366684157480811)),
 ('veut', np.float64(0.0640095430253574)),
 ('arriver temps', np.float64(0.06373512147389235)),
 ('cachée', np.float64(0.06373512147389235)),
 ('devait', np.float64(0.06373512147389235)),
 ('instant attention', np.float64(0.06373512147389235)),
 ('ménage', np.float64(0.06373512147389235))]

In [ ]:
# remove bad topic
df = df[df["topic"]!=5]

,start,stop,text,main_speaker,main_g,audio_file,ID,g_ath,topic,topic_prob
0,0.00,35.64,'est la question,SPEAKER_03,male,BIF_wav.wav,BIF,F,32,0.254804
1,0.00,35.64,a des favorites,SPEAKER_03,male,BIF_wav.wav,BIF,F,-1,0.052038
2,0.00,35.64,problème c'est de pouvoir confirmer le jour,SPEAKER_03,male,BIF_wav.wav,BIF,F,14,0.384110
3,0.00,35.64,aujourd'hui est un jour important pour les ath...,SPEAKER_03,male,BIF_wav.wav,BIF,F,-1,0.022161
4,0.00,35.64,qui a fait notamment un podium en ouverture de...,SPEAKER_03,male,BIF_wav.wav,BIF,F,-1,0.062766
...,...,...,...,...,...,...,...,...,...,...
8839,2893.44,2919.36,il y a - qui est là,SPEAKER_02,male,BMH_wav.wav,BMH,H,32,1.000000
8840,2893.44,2919.36,- on va garder ce chiffre en tête en tête à r...,SPEAKER_02,male,BMH_wav.wav,BMH,H,30,0.229441
8841,2893.44,2919.36,devient de toute l'histoire des le français l...,SPEAKER_02,male,BMH_wav.wav,BMH,H,-1,0.120464
8842,2893.44,2919.36,sacré cours et sacré jeu pour,SPEAKER_02,male,BMH_wav.wav,BMH,H,35,0.079278


Label count per topic

In [ ]:
# Only classified docs
clf = df[df["topic"]!=-1]
cdf = pd.DataFrame()
for i in range(len(clf["topic"].unique())) :
    tmp = df[df["topic"]==i]
    tmp = tmp["g_ath"].value_counts(normalize=True)
    tmp = pd.DataFrame([{
    "topic": f"topic_{i}",
    "M": tmp.get("M", 0),
    "F": tmp.get("F", 0),
    "H": tmp.get("H", 0)
}])
    cdf = pd.concat([cdf,tmp])

# All topics
tmp = clf["g_ath"].value_counts(normalize=True)
tmp = pd.DataFrame([{
    "topic": "all_topics",
    "M": tmp.get("M", 0),
    "F": tmp.get("F", 0),
    "H": tmp.get("H", 0)
}])
cdf = pd.concat([cdf,tmp])

for col in ["M", "F", "H"]:
    ref = tmp[col]
    cdf[f"{col}_diff"] = cdf[col] - ref
    # Format as +1.0%, -2.5% etc, leave all_topics row empty
    cdf[f"{col}_diff"] = cdf[f"{col}_diff"].apply(
        lambda x: f"{x*100:+.1f}%" if pd.notna(x) else ""
    )

cdf


,topic,M,F,H
0,topic_0,0.065744,0.422145,0.512111
0,topic_1,0.098540,0.408759,0.492701
0,topic_2,0.099174,0.388430,0.512397
0,topic_3,0.089362,0.319149,0.591489
0,topic_4,0.125000,0.370690,0.504310
0,topic_5,0.152542,0.310734,0.536723
0,topic_6,0.107784,0.383234,0.508982
0,topic_7,0.163636,0.333333,0.503030
0,topic_8,0.141935,0.432258,0.425806
0,topic_9,0.020979,0.475524,0.503497


Tests de stabilité (GPU nécessaire)

In [15]:
# Main words of current model themes

reference = set()
for topic_id in topic_model.get_topics():
    if topic_id == -1:
        continue
    words = [w for w, _ in topic_model.get_topic(topic_id)[:10]]
    reference.add(tuple(sorted(words)))
reference

{('1',
  '8',
  'attention',
  'bonsoir',
  'fantastique',
  'ok',
  'oui',
  'oui oui',
  'oui thème',
  'thème'),
 ('102', '102 55', '46', '54', '55', '62', '64', '70', '92', 'mesure'),
 ('2022',
  '2023',
  '2024',
  '2025',
  'champion',
  'champion monde',
  'europe',
  'monde',
  'monde 2023',
  'médailles'),
 ('23',
  'applaudissements',
  'attention',
  'bascule',
  'limite',
  'maxime',
  'maximus',
  'parfait',
  'suspense',
  'également'),
 ('2ème',
  '2ème note',
  'artistique',
  'monte',
  'monter',
  'note',
  'note artistique',
  'note note',
  'notes',
  'scorez'),
 ('4 tours',
  'axel tours',
  'demi',
  'seconde',
  'tour',
  'tour tours',
  'tours',
  'tours demi',
  'tours tour',
  'tours tours'),
 ('5',
  '5 points',
  'bonus',
  'bonus malus',
  'donner',
  'juges',
  'jury',
  'malus',
  'points',
  'élément'),
 ('71',
  '72',
  '78',
  '89',
  'meilleur',
  'meilleur score',
  'record',
  'saison',
  'score',
  'score saison'),
 ('accord',
  'complètement',
  '

On run 10 fois le même modèle à 80% du corpus pour voir la stabilité des thèmes

In [16]:
from sklearn.utils import resample
from collections import Counter

# List of words for each topic for each model
all_topics_words = []

# Number of runs
nb_iter = 10

for i in range(nb_iter):

    # First, resample 80% of docs
    sample = resample(docs, n_samples=int(len(docs) * 0.8), random_state=i)

    # Same model specification
    model = BERTopic(
    embedding_model=embedding_model,
    vectorizer_model=vectorizer_model,
    hdbscan_model=hdbscan_model,
    language="french",
    calculate_probabilities=True,
    verbose=True,
    seed_topic_list=seed    
)
    model.fit(sample)
    
    # Top 10 words per topic
    top_words = set()
    for topic_id in model.get_topics():
        if topic_id == -1:
            continue
        words = [w for w, _ in model.get_topic(topic_id)[:10]]
        top_words.add(tuple(sorted(words)))
    
    all_topics_words.append(top_words)


# Jaccard similarity index
def jaccard(t1, t2):
    s1, s2 = set(t1), set(t2)
    return len(s1 & s2) / len(s1 | s2)

# Topic stability
"""
Measures similarity between reference topic and topics found in other runs.

Thresholds for stability are determined by Jaccard index (threshold) and number of occurences (min_run)
"""
def is_stable(topic, all_runs, threshold=0.2, min_runs=7):
    count = 0
    for run_topics in all_runs:
        if any(jaccard(topic, t) >= threshold for t in run_topics):
            count += 1
    return count >= min_runs


stable_topics = [t for t in reference if is_stable(t, all_topics_words)]

print(f"Thèmes stables (≥7 runs sur 10) : {len(stable_topics)}")
for t in stable_topics:
    print(t)


2026-03-22 16:11:13,532 - BERTopic - Embedding - Transforming documents to embeddings.


Batches: 100%|██████████| 197/197 [00:09<00:00, 21.72it/s]
2026-03-22 16:11:22,678 - BERTopic - Embedding - Completed ✓
2026-03-22 16:11:22,679 - BERTopic - Guided - Find embeddings highly related to seeded topics.
Batches: 100%|██████████| 1/1 [00:00<00:00, 85.72it/s]
2026-03-22 16:11:22,744 - BERTopic - Guided - Completed ✓
2026-03-22 16:11:22,747 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-03-22 16:12:00,537 - BERTopic - Dimensionality - Completed ✓
2026-03-22 16:12:00,539 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-03-22 16:12:01,352 - BERTopic - Cluster - Completed ✓
2026-03-22 16:12:01,361 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-03-22 16:12:01,520 - BERTopic - Representation - Completed ✓
2026-03-22 16:12:01,726 - BERTopic - Embedding - Transforming documents to embeddings.
Batches: 100%|██████████| 197/197 [00:09<00:00, 21.73it/s]
2026-03-22 16:12:10,860 - BERTopic - Embeddin

Thèmes stables (≥7 runs sur 10) : 21
('2ème', '2ème note', 'artistique', 'monte', 'monter', 'note', 'note artistique', 'note note', 'notes', 'scorez')
('air', 'axel', 'combinaison', 'niveau', 'petit', 'points', 'saut', 'sauts', 'technique', 'triple')
('copie danse', 'danse', 'danse glace', 'discipline', 'feu', 'glace', 'glace couple', 'milan', 'patinoire', 'référence')
('boléro', 'choix', 'chorégraphie', 'chorégraphique', 'danse', 'interprétation', 'interprété', 'musical', 'musique', 'rythme')
('beaudry', 'cizeron', 'fournier', 'geoffrey', 'guillaume', 'guillaume cizeron', 'guillaume laurence', 'laurence', 'laurence fournier', 'laurence guillaume')
('5', '5 points', 'bonus', 'bonus malus', 'donner', 'juges', 'jury', 'malus', 'points', 'élément')
('court', 'libre', 'libre programme', 'meilleur programme', 'programme', 'programme court', 'programme libre', 'programme programme', 'programmes', 'voir programme')
('maximum', 'maximum niveau', 'niveau', 'niveau 4', 'niveau maximum', 'parfait

TODO ajouter la stabilité des labels 

keybertopic, extrait phrases saillantes
tester lda
virer thèmes nuls et rerun
checker distance cosine